# momentum-buffer-update composite — cx10: momentum buffer update via .copy_() for literal in-place semantics

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `momentum-buffer-update`, `buffer-copy_-inplace`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "momentum-buffer-update"
DD_ATOM_IDS = ["momentum-buffer-update", "buffer-copy_-inplace"]
DD_SUBTOPICS = ["Optimizer: Momentum buffer", "PyTorch: in-place buffer copy"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

The momentum recurrence `v <- mu * v + g` can be written three ways. Two are correct:

- `v.mul_(mu).add_(g)` — fused, no temporary.
- `v.copy_(mu * v + g)` — allocates a temporary for `mu*v + g`, then `copy_` writes it INTO the buffer's existing storage.

And one is WRONG:
- `v = mu * v + g` — rebinds the LOCAL `v`. The caller's list element is untouched.

This drill picks the second form on purpose: `.copy_()` makes the in-place semantics LITERAL — the buffer's storage receives the new value verbatim. ARENA's hand-rolled SGD uses this form because the comment ('this does need to be inplace, since we're modifying the value in self.b') stays true to the code.

**Anatomy.**
```python
for p, g, v in zip(params, grads, velocity):
    new_v = mu * v + g                # NEW tensor, not yet bound to state.
    v.copy_(new_v)                    # WRITE into v's storage; state[p] sees it.
    # ... param update happens elsewhere.
```

**Why both atoms together.** `.copy_()` IS the mechanism by which `momentum-buffer-update` becomes stateful. Pull the `copy_` out and you have a math expression that recomputes the same thing every step but never carries state forward.

### Composite Exercise — momentum buffer update via .copy_() for literal in-place semantics

**Atoms exercised together**: `momentum-buffer-update`, `buffer-copy_-inplace`

Implement `cx10_momentum_buffer_via_copy(grads, velocity_buffers, mu)`.

For each `(g, v)` pair:

1. Compute the new velocity `new_v = mu * v + g` as a fresh tensor.
2. Write it into the existing buffer with `v.copy_(new_v)`. **Use `.copy_()` explicitly** — the test inspects which approach you used by checking that the buffer's storage pointer is preserved AND that you didn't rely on the fused `mul_(mu).add_(g)` pattern (we test with a buffer aliased to an unrelated tensor; if you mutate via `mul_`, the alias diverges; if you `copy_`, the value lands but the alias is also updated — same storage).

Return the LIST of updated buffer references (so callers can use them as effective gradients).

Test runs THREE consecutive steps to verify momentum accumulates correctly and the buffer object identity is stable.

In [ ]:
def cx10_momentum_buffer_via_copy(grads, velocity_buffers, mu):
    out = []
    for g, v in zip(grads, velocity_buffers):
        # Atom A (momentum-buffer-update): v <- mu*v + g.
        new_v = mu * v + g
        # Atom B (buffer-copy_-inplace): write new_v INTO v's existing storage.
        v.copy_(new_v)
        out.append(v)
    return out


<details><summary>Show solution — cx10</summary>

```python
def cx10_momentum_buffer_via_copy(grads, velocity_buffers, mu):
    out = []
    for g, v in zip(grads, velocity_buffers):
        # Atom A (momentum-buffer-update): v <- mu*v + g.
        new_v = mu * v + g
        # Atom B (buffer-copy_-inplace): write new_v INTO v's existing storage.
        v.copy_(new_v)
        out.append(v)
    return out
```

Using `.copy_()` is slightly less efficient than `v.mul_(mu).add_(g)` (one extra allocation) but it's more LITERAL — the code reads exactly like the math: 'compute the new value, then put it into the buffer.' For PyTorch optimizer internals, both forms are used in the wild; ARENA picks `.copy_()` for pedagogical clarity. Either way, the key is that the storage at `velocity_buffers[i]` IS the new value when the function returns.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx10'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx10',
        'subtopics': ["Optimizer: Momentum buffer", "PyTorch: in-place buffer copy"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()